Here, we train scLEMBAS on the same split as Notebook 05B, but with different seeds to create an ensemble of learned models. We create an additional 4 per split in the 5-fold CV, giving a total of 25 models (5 of those originally trained in Notebook 05B and 20 from this split).

In [14]:
import os
import itertools

import numpy as np
import pandas as pd

import torch 

import sys
sclembas = '/home/hmbaghda/Projects/scLEMBAS'
sys.path.insert(1, os.path.join(sclembas))
from scLEMBAS import io
import scLEMBAS.utilities as utils

sys.path.insert(1, '../../.')
from McCauley_utils import initialize_mod_and_trainer, all_data

sys.path.insert(1, '../../../.') 
from notebook_utils import get_split

In [4]:
seed = 888
data_path = '/home/hmbaghda/orcd/pool/scLEMBAS/analysis'
author = 'McCauley'

n_cores = 30
os.environ["OMP_NUM_THREADS"] = str(1)
os.environ["MKL_NUM_THREADS"] = str(1)
os.environ["OPENBLAS_NUM_THREADS"] = str(1)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(1)
os.environ["NUMEXPR_NUM_THREADS"] = str(1)

In [5]:
(sn_ppis, tf_adata, adata, expr, source_label, target_label, weight_label, 
 stimulation_label, inhibition_label, cat_col, pert_col, ctrl_pert) = all_data


In [6]:
# 1% of real
num_stochastic_edges = int(np.round(0.01*sn_ppis.shape[0]))

## 1% of real + fake
# num_real_edges = sn_ppis.shape[0]
# num_stochastic_edges = (0.01*num_real_edges)/0.99
# num_stochastic_edges = int(np.round*num_stochastic_edges)
print('We will add {} stochastic edges, representing 1% of the edges in the PKN'.format(num_stochastic_edges))



We will add 176 stochastic edges, representing 1% of the edges in the PKN


Train an ensemble of 5 models per fold (25 total), each model containing a new set of stochastic edges representing 1% of the total number of true edges: 

In [5]:
n_folds = 5
n_ensembles_per_seed = 5
seed_multiplier = 21234

In [18]:
def write_model(mod, trainer, ensemble_idx, fn_base):
    # model and trainer
    io.write_pickled_object(trainer, fn_base + '_pruning_trainer_actual_ensemble{}.pickle'.format(ensemble_idx))
    torch.save(mod.state_dict(), fn_base + '_pruning_model_actual_ensemble{}.pt'.format(ensemble_idx))

    # stats
    trainer.stats['train'].to_csv(fn_base + '_pruning_trainstats_actual_ensemble{}.csv'.format(ensemble_idx))

    train_eval_df = trainer.stats['train_eval'].copy()
    test_df = trainer.stats['test'].copy()
    train_eval_df['loss_type'] = 'Train (Evaluation)'
    test_df['loss_type'] = 'Test'
    eval_df = pd.concat([train_eval_df, test_df], axis = 0)
    eval_df.reset_index(drop = True, inplace = True)
    eval_df.loss_type = pd.Categorical(eval_df.loss_type, ordered = True, 
                                      categories = ['Train (Evaluation)', 'Test'])
    eval_df.to_csv(fn_base + '_pruning_evalstats_actual_ensemble{}.csv'.format(ensemble_idx))
    
    del mod, trainer
    utils.clear_memory()

## L2 regularization

In [ ]:
bn_weights_l2s = [
    1e-7, # default
    1e-4, # worked on all bulk LEMBAS datasets
    1e-1
]
bn_weight_l1 = 0
iterations_ = itertools.product(range(n_folds), range(n_ensembles_per_seed), bn_weights_l2s)

for (fold, ensemble_idx, bn_weight_l2) in iterations_:
    l2_string = '{:.0E}'.format(bn_weight_l2).replace('E-0', 'E-')
    fn_base = os.path.join(data_path, 'processed', 'pruning_ensembles', '{}_fold{}_l2{}_l10'.format(author, fold, l2_string))


    for ensemble_idx in range(n_ensembles_per_seed):
        curr_seed = seed + ensemble_idx + 1 + (seed_multiplier * ensemble_idx * fold)        
        if os.path.isfile(fn_base + '_pruning_model_actual_ensemble{}.pt'.format(ensemble_idx)):
            continue
        mod, trainer = initialize_mod_and_trainer(
            fold = fold, 
            adversarial_penalty = True, 
            randomize = False, 
            num_stochastic_edges = num_stochastic_edges, 
            bn_weights_lambda_L2 = bn_weight_l2,
            bn_weights_lambda_L1 = bn_weight_l1, # 0
            seed = curr_seed)
        mod = trainer.train_model(verbose = False)
        write_model(mod, trainer, ensemble_idx = ensemble_idx, fn_base = fn_base)

## L1 regularization

In [ ]:
bn_weights_l1s = [
    1e-7, 1e-5, 1e-4, 1e-3,
]
bn_weight_l2 = 0

iterations_ = itertools.product(range(n_folds), range(n_ensembles_per_seed), bn_weights_l1s)


for (fold, ensemble_idx, bn_weight_l1) in iterations_:
    l1_string = '{:.0E}'.format(bn_weight_l1).replace('E-0', 'E-')
    fn_base = os.path.join(data_path, 'processed', 'pruning_ensembles', '{}_fold{}_l20_l1{}'.format(author, fold, l1_string))

    for ensemble_idx in range(n_ensembles_per_seed):
        curr_seed = seed + ensemble_idx + 1 + (seed_multiplier * ensemble_idx * fold)        
        if os.path.isfile(fn_base + '_pruning_model_actual_ensemble{}.pt'.format(ensemble_idx)):
            continue
        mod, trainer = initialize_mod_and_trainer(
            fold = fold, 
            adversarial_penalty = True, 
            randomize = False, 
            num_stochastic_edges = num_stochastic_edges, 
            bn_weights_lambda_L2 = bn_weight_l2, 
            bn_weights_lambda_L1 = bn_weight_l1, # 0
            seed = curr_seed)
        mod = trainer.train_model(verbose = False)
        write_model(mod, trainer, ensemble_idx = ensemble_idx, fn_base = fn_base)

In [22]:
# seed_multiplier = 21234
# def load_model(fold, ensemble_idx, bn_weight_l2 = 1e-7, bn_weight_l1 = 0, from_trainer = False):
#     """Loads the model and training object.

#     Two different ways to do so: from pickled training object (larger files) or from model state dict `.pt` file (smaller files to transfer).

#     Parameters
#     ----------
#     fold : int
#         fold split
#     ensemble_idx : int
#         ensemble index
#     from_trainer : bool, optional
#         whether to load from trainer object or model state dict, by default False
#         if False, the training object is not returned
#     """
#     curr_seed = seed + ensemble_idx + 1 + (seed_multiplier * ensemble_idx * fold)
    
#     l2_string = '{:.0E}'.format(bn_weight_l2).replace('E-0', 'E-') if bn_weight_l2 != 0 else '0'
#     l1_string = '{:.0E}'.format(bn_weight_l1).replace('E-0', 'E-') if bn_weight_l1 != 0 else '0'
#     fn_base = os.path.join(data_path, 'processed', 'pruning_ensembles', '{}_fold{}_l2{}_l1{}'.format(author, fold, l2_string, l1_string))
    
#     if from_trainer:
#         fn_trainer =  os.path.join(fn_base + '_pruning_trainer_actual_ensemble{}.pickle'.format(ensemble_idx))
#         trainer = io.read_pickled_object(fn_trainer)
#         mod = trainer.mod
#     else:
#         seed_ = seed + ensemble_idx + 1 + (seed_multiplier * ensemble_idx * fold) if ensemble_idx <= 3 else seed
#         mod, trainer = initialize_mod_and_trainer(
#             fold = fold, 
#             adversarial_penalty = True, 
#             randomize = False, 
#             num_stochastic_edges = num_stochastic_edges,
#             seed = curr_seed, 
#         )
        
#         fn_mod = os.path.join(fn_base + '_pruning_model_actual_ensemble{}.pt'.format(ensemble_idx))
#         mod.load_state_dict(torch.load(fn_mod))
#         trainer = None
#     return mod, trainer, fn_base

# from tqdm import tqdm
# bn_weights_l1s = [
#     1e-3, 1e-7, 1e-5, 
# ]
# bn_weight_l2 = 0
# iterations_ = itertools.product(bn_weights_l1s, range(5), range(5))

# for (bn_weight_l1, fold, ensemble_idx) in tqdm(iterations_):
#         l2_string = '{:.0E}'.format(bn_weight_l2).replace('E-0', 'E-') if bn_weight_l2 != 0 else '0'
#         l1_string = '{:.0E}'.format(bn_weight_l1).replace('E-0', 'E-') if bn_weight_l1 != 0 else '0'
#         fn_base = os.path.join(data_path, 'processed', 'pruning_ensembles', '{}_fold{}_l2{}_l1{}'.format(author, fold, l2_string, l1_string))
    
#         if os.path.isfile(fn_base + '_pruning_evalstats_actual_ensemble{}.csv'.format(ensemble_idx)):
#             continue
        
#         try:
#             mod, trainer, fn_base = load_model(fold, ensemble_idx, bn_weight_l2 = 0, bn_weight_l1 = 1e-3, 
#                                                from_trainer = True)

#             trainer.stats['train'].to_csv(fn_base + '_pruning_trainstats_actual_ensemble{}.csv'.format(ensemble_idx))

#             train_eval_df = trainer.stats['train_eval'].copy()
#             test_df = trainer.stats['test'].copy()
#             train_eval_df['loss_type'] = 'Train (Evaluation)'
#             test_df['loss_type'] = 'Test'
#             eval_df = pd.concat([train_eval_df, test_df], axis = 0)
#             eval_df.reset_index(drop = True, inplace = True)
#             eval_df.loss_type = pd.Categorical(eval_df.loss_type, ordered = True, 
#                                               categories = ['Train (Evaluation)', 'Test'])
#             eval_df.to_csv(fn_base + '_pruning_evalstats_actual_ensemble{}.csv'.format(ensemble_idx))
#             del mod, trainer
#             utils.clear_memory()
#         except:
#             pass
        
        
# bn_weights_l2s = [
#     1e-7, # default
#     1e-4, # worked on all bulk LEMBAS datasets
#     1e-1
# ]
# bn_weight_l1 = 0
# iterations_ = itertools.product(bn_weights_l1s, range(5), range(5))

# for (bn_weight_l1, fold, ensemble_idx) in tqdm(iterations_):
#         l2_string = '{:.0E}'.format(bn_weight_l2).replace('E-0', 'E-') if bn_weight_l2 != 0 else '0'
#         l1_string = '{:.0E}'.format(bn_weight_l1).replace('E-0', 'E-') if bn_weight_l1 != 0 else '0'
#         fn_base = os.path.join(data_path, 'processed', 'pruning_ensembles', '{}_fold{}_l2{}_l1{}'.format(author, fold, l2_string, l1_string))
    
#         if os.path.isfile(fn_base + '_pruning_evalstats_actual_ensemble{}.csv'.format(ensemble_idx)):
#             continue
        
#         try:
#             mod, trainer, fn_base = load_model(fold, ensemble_idx, bn_weight_l2 = 0, bn_weight_l1 = 1e-3, 
#                                                from_trainer = True)

#             trainer.stats['train'].to_csv(fn_base + '_pruning_trainstats_actual_ensemble{}.csv'.format(ensemble_idx))

#             train_eval_df = trainer.stats['train_eval'].copy()
#             test_df = trainer.stats['test'].copy()
#             train_eval_df['loss_type'] = 'Train (Evaluation)'
#             test_df['loss_type'] = 'Test'
#             eval_df = pd.concat([train_eval_df, test_df], axis = 0)
#             eval_df.reset_index(drop = True, inplace = True)
#             eval_df.loss_type = pd.Categorical(eval_df.loss_type, ordered = True, 
#                                               categories = ['Train (Evaluation)', 'Test'])
#             eval_df.to_csv(fn_base + '_pruning_evalstats_actual_ensemble{}.csv'.format(ensemble_idx))
#             del mod, trainer
#             utils.clear_memory()
#         except:
#             pass
    